**Repository release note (2026-10-11):** Controlled full-course VLE and eligible assessment feature construction, matched to leakage-free registration keys and outcome labels.

Saved execution outputs were removed from this public copy; source calculations have not been independently rerun. Configure `OULAD_PROJECT_DIR` or use the original Colab directory.


In [ ]:
import pandas as pd
import numpy as np
import os

BASE_PATH = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

DATA_PATH = os.path.join(BASE_PATH, "data")
PROCESSED_PATH = os.path.join(BASE_PATH, "processed")
MODEL_DATA_PATH = os.path.join(PROCESSED_PATH, "model_datasets")
RESULTS_PATH = os.path.join(BASE_PATH, "results")

LEAKY_PATH = os.path.join(
    PROCESSED_PATH,
    "leaky_datasets"
)

os.makedirs(LEAKY_PATH, exist_ok=True)

keys = [
    "code_module",
    "code_presentation",
    "id_student"
]

print("Notebook 04 environment ready.")
print("Data folder:", os.path.exists(DATA_PATH))
print("Model datasets folder:", os.path.exists(MODEL_DATA_PATH))
print("Leaky output folder:", os.path.exists(LEAKY_PATH))

In [ ]:
for p in [10, 20, 30, 40]:

    path = os.path.join(
        MODEL_DATA_PATH,
        f"leakage_free_{p}.csv"
    )

    print(
        f"{p}% leakage-free dataset:",
        os.path.exists(path)
    )

In [ ]:
lf_10 = pd.read_csv(
    os.path.join(
        MODEL_DATA_PATH,
        "leakage_free_10.csv"
    )
)

print("Rows:", len(lf_10))
print(
    "Unique students:",
    lf_10["id_student"].nunique()
)

print("\nTarget distribution:")
print(
    lf_10["final_result"]
    .value_counts()
)

In [ ]:
lf_datasets = {}

for p in [10, 20, 30, 40]:
    lf_datasets[p] = pd.read_csv(
        os.path.join(
            MODEL_DATA_PATH,
            f"leakage_free_{p}.csv"
        )
    )

all_cohort_keys = pd.concat(
    [
        df[keys]
        for df in lf_datasets.values()
    ],
    ignore_index=True
).drop_duplicates()

print(
    "Unique registration records needed:",
    f"{len(all_cohort_keys):,}"
)

In [ ]:
registration = pd.read_csv(
    os.path.join(
        DATA_PATH,
        "studentRegistration.csv"
    )
)

courses = pd.read_csv(
    os.path.join(
        DATA_PATH,
        "courses.csv"
    )
)

full_vle_lookup = (
    all_cohort_keys
    .merge(
        registration[
            keys + ["date_registration"]
        ],
        on=keys,
        how="left"
    )
    .merge(
        courses,
        on=[
            "code_module",
            "code_presentation"
        ],
        how="left"
    )
)

print("Lookup rows:", len(full_vle_lookup))

print("\nMissing timing information:")
print(
    full_vle_lookup[
        [
            "date_registration",
            "module_presentation_length"
        ]
    ].isna().sum()
)

In [ ]:
vle_info = pd.read_csv(
    os.path.join(
        DATA_PATH,
        "vle.csv"
    )
)

vle_map = vle_info[
    [
        "code_module",
        "code_presentation",
        "id_site",
        "activity_type"
    ]
].drop_duplicates()

activity_group_map = {
    "oucontent": "content",
    "subpage": "content",
    "resource": "content",
    "forumng": "forum",
    "homepage": "navigation"
}

student_vle_path = os.path.join(
    DATA_PATH,
    "studentVle.csv"
)

print("VLE activity mapping ready.")

In [ ]:
full_click_parts = []
full_day_parts = []
full_site_parts = []
full_group_parts = []

total_matched_rows = 0
future_course_rows_excluded = 0
preregistration_rows_excluded = 0

for chunk in pd.read_csv(
    student_vle_path,
    chunksize=500_000
):

    merged = chunk.merge(
        full_vle_lookup,
        on=keys,
        how="inner"
    )

    total_matched_rows += len(merged)

    # Audit only — these are data-quality exclusions,
    # NOT our experimental temporal cutoff.
    preregistration_rows_excluded += (
        merged["date"] <
        merged["date_registration"]
    ).sum()

    future_course_rows_excluded += (
        merged["date"] >
        merged["module_presentation_length"]
    ).sum()

    # Full-course leaky window:
    # course start through official course end.
    full = merged[
        (merged["date"] >= 0) &
        (
            merged["date"] >=
            merged["date_registration"]
        ) &
        (
            merged["date"] <=
            merged["module_presentation_length"]
        )
    ].copy()

    if len(full) == 0:
        continue

    # Total clicks
    full_click_parts.append(
        full.groupby(
            keys,
            as_index=False
        )["sum_click"].sum()
    )

    # Unique active days
    full_day_parts.append(
        full[
            keys + ["date"]
        ].drop_duplicates()
    )

    # Unique VLE sites
    full_site_parts.append(
        full[
            keys + ["id_site"]
        ].drop_duplicates()
    )

    # Activity groups
    typed = full.merge(
        vle_map,
        on=[
            "code_module",
            "code_presentation",
            "id_site"
        ],
        how="left"
    )

    typed["activity_group"] = (
        typed["activity_type"]
        .map(activity_group_map)
    )

    typed = typed[
        typed["activity_group"].notna()
    ]

    if len(typed) > 0:
        full_group_parts.append(
            typed.groupby(
                keys + ["activity_group"],
                as_index=False
            )["sum_click"].sum()
        )

print("Full-course VLE scan completed.")
print(
    "Matched VLE rows:",
    f"{total_matched_rows:,}"
)
print(
    "Pre-registration rows excluded:",
    f"{preregistration_rows_excluded:,}"
)
print(
    "After-course rows excluded:",
    f"{future_course_rows_excluded:,}"
)

In [ ]:
full_clicks = (
    pd.concat(
        full_click_parts,
        ignore_index=True
    )
    .groupby(
        keys,
        as_index=False
    )["sum_click"]
    .sum()
    .rename(
        columns={
            "sum_click": "vle_clicks_full"
        }
    )
)

full_days = (
    pd.concat(
        full_day_parts,
        ignore_index=True
    )
    .drop_duplicates()
    .groupby(keys)
    .size()
    .reset_index(
        name="vle_active_days_full"
    )
)

full_sites = (
    pd.concat(
        full_site_parts,
        ignore_index=True
    )
    .drop_duplicates()
    .groupby(keys)
    .size()
    .reset_index(
        name="vle_unique_sites_full"
    )
)

print("General full-course VLE features created.")

In [ ]:
full_group_clicks = pd.concat(
    full_group_parts,
    ignore_index=True
)

full_group_clicks = (
    full_group_clicks
    .groupby(
        keys + ["activity_group"],
        as_index=False
    )["sum_click"]
    .sum()
)

full_group_clicks = (
    full_group_clicks
    .pivot_table(
        index=keys,
        columns="activity_group",
        values="sum_click",
        fill_value=0
    )
    .reset_index()
)

full_group_clicks.columns.name = None

full_group_clicks = (
    full_group_clicks.rename(
        columns={
            "content": "content_clicks_full",
            "forum": "forum_clicks_full",
            "navigation": "navigation_clicks_full"
        }
    )
)

full_group_clicks.head()

In [ ]:
vle_full = all_cohort_keys.copy()

for table in [
    full_clicks,
    full_days,
    full_sites,
    full_group_clicks
]:

    vle_full = vle_full.merge(
        table,
        on=keys,
        how="left"
    )

full_feature_cols = [
    "vle_clicks_full",
    "vle_active_days_full",
    "vle_unique_sites_full",
    "content_clicks_full",
    "forum_clicks_full",
    "navigation_clicks_full"
]

vle_full[full_feature_cols] = (
    vle_full[full_feature_cols]
    .fillna(0)
    .astype(int)
)

print("Full-course VLE table created.")
print("Rows:", f"{len(vle_full):,}")
print(
    "Duplicates:",
    vle_full.duplicated(subset=keys).sum()
)

print("\nMissing values:")
print(
    vle_full[
        full_feature_cols
    ].isna().sum()
)

In [ ]:
compare_10 = (
    lf_datasets[10][
        keys + [
            "vle_clicks_10",
            "vle_active_days_10",
            "vle_unique_sites_10",
            "content_clicks_10",
            "forum_clicks_10",
            "navigation_clicks_10"
        ]
    ]
    .merge(
        vle_full,
        on=keys,
        how="left"
    )
)

checks = {
    "Total clicks":
        compare_10["vle_clicks_full"]
        >= compare_10["vle_clicks_10"],

    "Active days":
        compare_10["vle_active_days_full"]
        >= compare_10["vle_active_days_10"],

    "Unique sites":
        compare_10["vle_unique_sites_full"]
        >= compare_10["vle_unique_sites_10"],

    "Content clicks":
        compare_10["content_clicks_full"]
        >= compare_10["content_clicks_10"],

    "Forum clicks":
        compare_10["forum_clicks_full"]
        >= compare_10["forum_clicks_10"],

    "Navigation clicks":
        compare_10["navigation_clicks_full"]
        >= compare_10["navigation_clicks_10"]
}

for name, condition in checks.items():
    print(
        name,
        "- violations:",
        (~condition).sum()
    )

In [ ]:
print(
    "Students with additional clicks after 10%:",
    (
        compare_10["vle_clicks_full"]
        >
        compare_10["vle_clicks_10"]
    ).sum()
)

print(
    "Percent gaining future VLE information:",
    round(
        (
            compare_10["vle_clicks_full"]
            >
            compare_10["vle_clicks_10"]
        ).mean() * 100,
        2
    )
)

In [ ]:
vle_full.to_csv(
    os.path.join(
        LEAKY_PATH,
        "vle_features_full_course.csv"
    ),
    index=False
)

print("Full-course leaky VLE features saved.")

In [ ]:
assessments = pd.read_csv(
    os.path.join(DATA_PATH, "assessments.csv")
)

student_assessment = pd.read_csv(
    os.path.join(DATA_PATH, "studentAssessment.csv")
)

assessment_full = student_assessment.merge(
    assessments,
    on="id_assessment",
    how="left"
)

assessment_full = assessment_full.merge(
    all_cohort_keys,
    on=keys,
    how="inner"
)

assessment_full = assessment_full.merge(
    registration[
        keys + ["date_registration"]
    ],
    on=keys,
    how="left"
)

assessment_full = assessment_full.merge(
    courses,
    on=[
        "code_module",
        "code_presentation"
    ],
    how="left"
)

print("Assessment records for experimental cohorts:",
      f"{len(assessment_full):,}")

In [ ]:
assessment_leaky = assessment_full[
    (assessment_full["is_banked"] == 0) &
    (
        assessment_full["assessment_type"]
        .isin(["TMA", "CMA"])
    ) &
    (
        assessment_full["date_submitted"] >=
        assessment_full["date_registration"]
    ) &
    (
        assessment_full["date_submitted"] <=
        assessment_full["module_presentation_length"]
    )
].copy()

print(
    "Non-banked TMA/CMA records retained:",
    f"{len(assessment_leaky):,}"
)

print(
    "Missing scores:",
    assessment_leaky["score"].isna().sum()
)

In [ ]:
assessment_leaky["late_flag"] = (
    assessment_leaky["date_submitted"] >
    assessment_leaky["date"]
).astype(int)

assessment_leaky["submission_delay"] = (
    assessment_leaky["date_submitted"] -
    assessment_leaky["date"]
)

timing_full = (
    assessment_leaky
    .groupby(keys)
    .agg(
        assessments_submitted_full=(
            "id_assessment", "nunique"
        ),
        tma_submitted_full=(
            "assessment_type",
            lambda x: (x == "TMA").sum()
        ),
        cma_submitted_full=(
            "assessment_type",
            lambda x: (x == "CMA").sum()
        ),
        late_submissions_full=(
            "late_flag", "sum"
        ),
        mean_submission_delay_full=(
            "submission_delay", "mean"
        )
    )
    .reset_index()
)

print("Full-course assessment timing features created.")

In [ ]:
score_full_source = assessment_leaky[
    assessment_leaky["score"].notna()
].copy()

score_full = (
    score_full_source
    .groupby(keys)
    .agg(
        scored_assessments_full=(
            "score", "count"
        ),
        mean_score_full=(
            "score", "mean"
        )
    )
    .reset_index()
)

print("Full-course score features created.")

In [ ]:
assessment_features_full = all_cohort_keys.copy()

assessment_features_full = (
    assessment_features_full
    .merge(
        timing_full,
        on=keys,
        how="left"
    )
    .merge(
        score_full,
        on=keys,
        how="left"
    )
)

count_cols_full = [
    "assessments_submitted_full",
    "tma_submitted_full",
    "cma_submitted_full",
    "late_submissions_full",
    "scored_assessments_full"
]

assessment_features_full[count_cols_full] = (
    assessment_features_full[count_cols_full]
    .fillna(0)
    .astype(int)
)

assessment_features_full["has_score_full"] = (
    assessment_features_full[
        "scored_assessments_full"
    ] > 0
).astype(int)

print("Full-course assessment table created.")
print("Rows:", f"{len(assessment_features_full):,}")

print(
    "Duplicates:",
    assessment_features_full
    .duplicated(subset=keys)
    .sum()
)

print("\nMissing values:")
missing_full = assessment_features_full.isna().sum()
print(missing_full[missing_full > 0])

In [ ]:
assessment_compare_10 = (
    lf_datasets[10][
        keys + [
            "assessments_submitted_10",
            "scored_assessments_10"
        ]
    ]
    .merge(
        assessment_features_full,
        on=keys,
        how="left"
    )
)

print(
    "Students gaining future assessment submissions:",
    (
        assessment_compare_10[
            "assessments_submitted_full"
        ] >
        assessment_compare_10[
            "assessments_submitted_10"
        ]
    ).sum()
)

print(
    "Percent gaining future assessment submissions:",
    round(
        (
            assessment_compare_10[
                "assessments_submitted_full"
            ] >
            assessment_compare_10[
                "assessments_submitted_10"
            ]
        ).mean() * 100,
        2
    )
)

print(
    "\nStudents gaining additional scored assessments:",
    (
        assessment_compare_10[
            "scored_assessments_full"
        ] >
        assessment_compare_10[
            "scored_assessments_10"
        ]
    ).sum()
)

print(
    "Percent gaining additional scored assessments:",
    round(
        (
            assessment_compare_10[
                "scored_assessments_full"
            ] >
            assessment_compare_10[
                "scored_assessments_10"
            ]
        ).mean() * 100,
        2
    )
)

In [ ]:
assessment_features_full.to_csv(
    os.path.join(
        LEAKY_PATH,
        "assessment_features_full_course.csv"
    ),
    index=False
)

print("Full-course leaky assessment features saved.")

In [ ]:
full_assessment_cols = [
    "assessments_submitted_full",
    "tma_submitted_full",
    "cma_submitted_full",
    "late_submissions_full",
    "mean_submission_delay_full",
    "scored_assessments_full",
    "mean_score_full",
    "has_score_full"
]

print("Rows:", len(assessment_features_full))
print(
    "Matches union cohort:",
    len(assessment_features_full) == len(all_cohort_keys)
)
print(
    "Duplicates:",
    assessment_features_full.duplicated(subset=keys).sum()
)

print("\nMissing values:")
print(
    assessment_features_full[
        full_assessment_cols
    ].isna().sum()
)

In [ ]:
def build_leaky_dataset(p):

    lf = pd.read_csv(
        os.path.join(
            MODEL_DATA_PATH,
            f"leakage_free_{p}.csv"
        )
    )

    vle_full_saved = pd.read_csv(
        os.path.join(
            LEAKY_PATH,
            "vle_features_full_course.csv"
        )
    )

    assessment_full_saved = pd.read_csv(
        os.path.join(
            LEAKY_PATH,
            "assessment_features_full_course.csv"
        )
    )

    # --------------------------------
    # Remove leakage-free in-course VLE
    # --------------------------------
    early_vle_cols = [
        f"vle_clicks_{p}",
        f"vle_active_days_{p}",
        f"vle_unique_sites_{p}",
        f"content_clicks_{p}",
        f"forum_clicks_{p}",
        f"navigation_clicks_{p}"
    ]

    # Pre-course VLE variables are NOT removed.
    # They are valid baseline information.

    # --------------------------------
    # Remove leakage-free assessment
    # variables
    # --------------------------------
    early_assessment_cols = [
        f"assessments_submitted_{p}",
        f"tma_submitted_{p}",
        f"cma_submitted_{p}",
        f"late_submissions_{p}",
        f"mean_submission_delay_{p}",
        f"scored_assessments_{p}",
        f"mean_score_due_{p}",
        f"has_score_due_{p}"
    ]

    remove_cols = (
        early_vle_cols +
        early_assessment_cols
    )

    missing_remove = [
        col for col in remove_cols
        if col not in lf.columns
    ]

    if missing_remove:
        raise ValueError(
            f"Missing expected columns at {p}%: "
            f"{missing_remove}"
        )

    leaky = lf.drop(
        columns=remove_cols
    ).copy()

    # --------------------------------
    # Add full-course VLE information
    # --------------------------------
    leaky = leaky.merge(
        vle_full_saved,
        on=keys,
        how="left",
        validate="one_to_one"
    )

    # --------------------------------
    # Add full-course assessment info
    # --------------------------------
    leaky = leaky.merge(
        assessment_full_saved,
        on=keys,
        how="left",
        validate="one_to_one"
    )

    print(f"\n=== LEAKY {p}% DATASET ===")
    print("Rows:", f"{len(leaky):,}")
    print("Columns:", len(leaky.columns))

    print(
        "Duplicates:",
        leaky.duplicated(
            subset=keys
        ).sum()
    )

    print(
        "Target unchanged:",
        leaky["final_result"]
        .value_counts()
        .sort_index()
        .equals(
            lf["final_result"]
            .value_counts()
            .sort_index()
        )
    )

    return leaky

In [ ]:
leaky_10 = build_leaky_dataset(10)

In [ ]:
leaky_20 = build_leaky_dataset(20)

In [ ]:
leaky_30 = build_leaky_dataset(30)

In [ ]:
leaky_40 = build_leaky_dataset(40)

In [ ]:
leaky_sets = {
    10: leaky_10,
    20: leaky_20,
    30: leaky_30,
    40: leaky_40
}

for p, df in leaky_sets.items():

    df.to_csv(
        os.path.join(
            LEAKY_PATH,
            f"leaky_{p}.csv"
        ),
        index=False
    )

print("All four controlled leaky datasets saved.")

In [ ]:
for p in [10, 20, 30, 40]:

    lf = lf_datasets[p]
    leaky = leaky_sets[p]

    lf_keys = set(
        map(
            tuple,
            lf[keys].to_numpy()
        )
    )

    leaky_keys = set(
        map(
            tuple,
            leaky[keys].to_numpy()
        )
    )

    print(f"\n--- {p}% ---")
    print(
        "Same registration records:",
        lf_keys == leaky_keys
    )

    target_check = (
        lf[
            keys + ["final_result"]
        ]
        .merge(
            leaky[
                keys + ["final_result"]
            ],
            on=keys,
            suffixes=("_lf", "_leaky")
        )
    )

    print(
        "Same targets:",
        (
            target_check["final_result_lf"]
            ==
            target_check["final_result_leaky"]
        ).all()
    )

In [ ]:
leakage_exposure = []

for p in [10, 20, 30, 40]:

    lf = lf_datasets[p]

    temp = (
        lf[
            keys + [
                f"vle_clicks_{p}",
                f"assessments_submitted_{p}",
                f"scored_assessments_{p}"
            ]
        ]
        .merge(
            vle_full[
                keys + ["vle_clicks_full"]
            ],
            on=keys,
            how="left"
        )
        .merge(
            assessment_features_full[
                keys + [
                    "assessments_submitted_full",
                    "scored_assessments_full"
                ]
            ],
            on=keys,
            how="left"
        )
    )

    leakage_exposure.append({
        "Cutoff": f"{p}%",
        "Students": len(temp),

        "Future_VLE_Percent": round(
            (
                temp["vle_clicks_full"]
                >
                temp[f"vle_clicks_{p}"]
            ).mean() * 100,
            2
        ),

        "Future_Assessment_Percent": round(
            (
                temp["assessments_submitted_full"]
                >
                temp[f"assessments_submitted_{p}"]
            ).mean() * 100,
            2
        ),

        "Future_Score_Percent": round(
            (
                temp["scored_assessments_full"]
                >
                temp[f"scored_assessments_{p}"]
            ).mean() * 100,
            2
        )
    })

leakage_exposure = pd.DataFrame(
    leakage_exposure
)

leakage_exposure

In [ ]:
leakage_exposure.to_csv(
    os.path.join(
        RESULTS_PATH,
        "leakage_exposure_by_cutoff.csv"
    ),
    index=False
)

print("Leakage exposure audit saved.")

In [ ]:
print("=== FINAL NOTEBOOK 04 CHECK ===")

print("\nLEAKAGE EXPOSURE BY CUTOFF")
print(leakage_exposure.to_string(index=False))

print("\n" + "=" * 65)

for p in [10, 20, 30, 40]:

    lf = lf_datasets[p]
    leaky = leaky_sets[p]

    lf_keys = set(
        map(tuple, lf[keys].to_numpy())
    )

    leaky_keys = set(
        map(tuple, leaky[keys].to_numpy())
    )

    target_check = (
        lf[keys + ["final_result"]]
        .merge(
            leaky[keys + ["final_result"]],
            on=keys,
            suffixes=("_lf", "_leaky")
        )
    )

    print(f"\n--- {p}% cutoff ---")
    print("Leakage-free rows:", len(lf))
    print("Leaky rows:", len(leaky))

    print(
        "Same registration records:",
        lf_keys == leaky_keys
    )

    print(
        "Same targets:",
        (
            target_check["final_result_lf"]
            ==
            target_check["final_result_leaky"]
        ).all()
    )

    print(
        "Leaky duplicates:",
        leaky.duplicated(subset=keys).sum()
    )

In [ ]:
for p in [30, 40]:
    lf = lf_datasets[p]
    leaky = leaky_sets[p]

    same_records = set(
        map(tuple, lf[keys].to_numpy())
    ) == set(
        map(tuple, leaky[keys].to_numpy())
    )

    target_check = lf[
        keys + ["final_result"]
    ].merge(
        leaky[keys + ["final_result"]],
        on=keys,
        suffixes=("_lf", "_leaky")
    )

    same_targets = (
        target_check["final_result_lf"]
        ==
        target_check["final_result_leaky"]
    ).all()

    print(f"{p}%:")
    print("Rows:", len(lf), len(leaky))
    print("Same records:", same_records)
    print("Same targets:", same_targets)
    print(
        "Duplicates:",
        leaky.duplicated(subset=keys).sum()
    )
    print()

In [ ]:
print(
    leakage_exposure[
        [
            "Cutoff",
            "Future_VLE_Percent",
            "Future_Assessment_Percent",
            "Future_Score_Percent"
        ]
    ].to_string(index=False)
)

In [ ]:
for _, row in leakage_exposure.iterrows():

    print(f"\n{row['Cutoff']} cutoff")
    print(
        "Future VLE:",
        row["Future_VLE_Percent"], "%"
    )
    print(
        "Future assessment submissions:",
        row["Future_Assessment_Percent"], "%"
    )
    print(
        "Future scores:",
        row["Future_Score_Percent"], "%"
    )